# RF-DETR Training on LaRS Dataset

This notebook demonstrates how to train RF-DETR segmentation model on the LaRS (Labeled Maritime Segmentation) dataset.

## Dataset Information
- **Dataset**: LaRS v1.0.0
- **Task**: Instance Segmentation
- **Categories**: Static Obstacle, Water, Sky, Boat/ship, Row boats, Paddle board, Buoy, Swimmer, Animal, Float, Other
- **Training images**: 2605
- **Validation images**: 198

In [1]:
import os
import sys
from pathlib import Path

# Add parent directory to path to import modules
sys.path.append(str(Path.cwd().parent))

print("Working directory:", Path.cwd())
print("Python version:", sys.version)

Working directory: /Users/emil/Documents/projects/rf-detr/notebooks
Python version: 3.11.5 (main, Sep 11 2023, 08:31:25) [Clang 14.0.6 ]


In [2]:
import json
from pathlib import Path

# Dataset paths
dataset_dir = Path("../datasets/lars_rfdetr")
train_annotations = dataset_dir / "train" / "_annotations.coco.json"
valid_annotations = dataset_dir / "valid" / "_annotations.coco.json"

# Load and inspect annotations
with open(train_annotations) as f:
    train_data = json.load(f)

with open(valid_annotations) as f:
    valid_data = json.load(f)

print("Dataset Structure:")
print(f"  Train images: {len(train_data['images'])}")
print(f"  Train annotations: {len(train_data['annotations'])}")
print(f"  Valid images: {len(valid_data['images'])}")
print(f"  Valid annotations: {len(valid_data['annotations'])}")
print(f"\nCategories ({len(train_data['categories'])}):")
for cat in train_data['categories']:
    print(f"  {cat['id']}: {cat['name']}")

Dataset Structure:
  Train images: 2605
  Train annotations: 16951
  Valid images: 198
  Valid annotations: 1667

Categories (11):
  1: Static Obstacle
  3: Water
  5: Sky
  11: Boat/ship
  12: Row boats
  13: Paddle board
  14: Buoy
  15: Swimmer
  16: Animal
  17: Float
  19: Other


In [3]:
import supervision as sv

# Load trained model
checkpoint_path = "../output/checkpoint_best_total.pth"
model = RFDETRSegPreview(pretrain_weights=checkpoint_path)
model.optimize_for_inference()

# Get a validation image
val_img_info = random.choice(valid_data['images'])
val_img_path = dataset_dir / "valid" / "images" / val_img_info['file_name']

# Load image
image = cv2.imread(str(val_img_path))

# Run inference
detections = model.predict(image, threshold=0.5)

# Visualize results
mask_annotator = sv.MaskAnnotator()
label_annotator = sv.LabelAnnotator()

annotated_image = mask_annotator.annotate(scene=image.copy(), detections=detections)
annotated_image = label_annotator.annotate(scene=annotated_image, detections=detections)

# Display
image_rgb = cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(15, 10))
plt.imshow(image_rgb)
plt.title(f"Predictions on: {val_img_info['file_name']}")
plt.axis('off')
plt.show()

print(f"Detected {len(detections)} objects")

NameError: name 'RFDETRSegPreview' is not defined

## 4. Run Inference on Test Image

In [ ]:
from rfdetr import RFDETRSegPreview

# Initialize model
model = RFDETRSegPreview()

# Training configuration
config = {
    "dataset_dir": str(dataset_dir),
    "epochs": 100,
    "batch_size": 4,
    "grad_accum_steps": 4,
    "lr": 1e-4,
    "output_dir": "../output"
}

print("Training Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")
print(f"\nEffective batch size: {config['batch_size'] * config['grad_accum_steps']}")

# Start training
print("\nStarting training...")
model.train(**config)

## 3. Train RF-DETR Model

In [ ]:
import cv2
import matplotlib.pyplot as plt
import random

# Get a random training image
sample_img_info = random.choice(train_data['images'])
img_path = dataset_dir / "train" / "images" / sample_img_info['file_name']

# Load and display image
image = cv2.imread(str(img_path))
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(image_rgb)
plt.title(f"Sample Image: {sample_img_info['file_name']}")
plt.axis('off')
plt.show()

print(f"Image ID: {sample_img_info['id']}")
print(f"Dimensions: {sample_img_info['width']}x{sample_img_info['height']}")

## 2. Visualize Sample Images

## 1. Verify Dataset Structure